# 02 - Experiences comparatives

Optimisation sequentielle greedy (on fige le meilleur a chaque etape).

- Etude A: 3 embedders (Mistral + prompt strict)
- Etude B: 3 LLMs (meilleur embedder + prompt strict)
- Etude C: 3 prompts (meilleur embedder + meilleur LLM)

In [ ]:
# sur colab:
# !pip install -q -r ../requirements.txt

import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline
from evaluation.evaluate import (
    load_dataset, retrieval_recall, keyword_score, measure_latency
)
import pandas as pd

dataset = load_dataset('../evaluation/qa_dataset.json')
print('nb questions:', len(dataset))

## Etude A - embedders

In [ ]:
CONFIGS_EMB = [
    ('minilm', 'sentence-transformers/all-MiniLM-L6-v2', 'collection_minilm'),
    ('e5_base', 'intfloat/multilingual-e5-base', 'collection_e5_base'),
    ('solon', 'OrdalieTech/Solon-embeddings-base-0.1', 'collection_solon_base'),
]

print('loading mistral (baseline)...')
llm = LLMGenerator('mistralai/Mistral-7B-Instruct-v0.3')

results_A = []
for name, model, coll in CONFIGS_EMB:
    print(f'\n--- {name} ---')
    emb = Embedder(model)
    vs = VectorStore('../data/chroma_db', coll)
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    r = retrieval_recall(pipe, dataset, k=4)
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_A.append({
        'embedder': name,
        'recall@4': round(r, 3),
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_A[-1])

df_A = pd.DataFrame(results_A)
print(df_A)
df_A.to_csv('../evaluation/results_A_embedders.csv', index=False)

## Etude B - LLMs

In [ ]:
best_row = df_A.sort_values('recall@4', ascending=False).iloc[0]
best_emb_name = best_row['embedder']
name_to_cfg = {n: (m, c) for (n, m, c) in CONFIGS_EMB}
best_model, best_coll = name_to_cfg[best_emb_name]
print('meilleur embedder:', best_emb_name)

emb = Embedder(best_model)
vs = VectorStore('../data/chroma_db', best_coll)

CONFIGS_LLM = [
    ('mistral7b', 'mistralai/Mistral-7B-Instruct-v0.3'),
    ('qwen7b', 'Qwen/Qwen2.5-7B-Instruct'),
    ('llama8b', 'meta-llama/Llama-3.1-8B-Instruct'),  # fallback: croissantllm/CroissantLLMChat-v0.1
]

results_B = []
for name, model in CONFIGS_LLM:
    print(f'\n--- {name} ---')
    try:
        del llm
    except NameError:
        pass
    gc.collect(); torch.cuda.empty_cache()
    try:
        llm = LLMGenerator(model)
    except Exception as e:
        print(f'skip {name}: {e}')
        continue
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_B.append({
        'llm': name,
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_B[-1])

df_B = pd.DataFrame(results_B)
print(df_B)
df_B.to_csv('../evaluation/results_B_llms.csv', index=False)

## Etude C - prompts

In [ ]:
best_llm_row = df_B.sort_values('keyword', ascending=False).iloc[0]
best_llm_name = best_llm_row['llm']
name_to_llm = {n: m for (n, m) in CONFIGS_LLM}
best_llm_model = name_to_llm[best_llm_name]
print('meilleur llm:', best_llm_name)

try:
    del llm
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
llm = LLMGenerator(best_llm_model)

results_C = []
examples_c = []
for prompt_name in ['strict', 'citation', 'structured']:
    print(f'\n--- prompt {prompt_name} ---')
    pipe = RAGPipeline(emb, vs, llm, prompt_name)
    k_score, details = keyword_score(pipe, dataset, k=4)
    results_C.append({'prompt': prompt_name, 'keyword': round(k_score, 3)})
    print(results_C[-1])
    # garde 2 exemples pour les slides
    for d in details[:2]:
        examples_c.append({'prompt': prompt_name, **d})

df_C = pd.DataFrame(results_C)
print(df_C)
df_C.to_csv('../evaluation/results_C_prompts.csv', index=False)
pd.DataFrame(examples_c).to_csv('../evaluation/examples_qualitatifs.csv', index=False)

## Recap - meilleure config

In [ ]:
best_prompt = df_C.sort_values('keyword', ascending=False).iloc[0]['prompt']
print('config retenue:')
print(f'  embedder: {best_emb_name}')
print(f'  llm: {best_llm_name}')
print(f'  prompt: {best_prompt}')
print('\nmettre a jour app.py avec ces valeurs avant la demo.')